## KNN(K-Nearest Neighbor) 검색


*   가장 유사한 K개 벡터 반환.
*   (+) 모든 데이터 조사 => 정확. (재현율 100%)
*   (-) 벡터 차원이 커지면 연산량이 비례하게 늘어나 속도가 느려짐.  



---
##### 용어 설명
* 재현율(recall): 가장 가까운 K개의 정답 중 검색 결과로 반환된 비율.


In [ ]:
!wget ftp://ftp.irisa.fr/local/texmex/corpus/sift.tar.gz
!tar -xf sift.tar.gz
!mkdir data/sift1M -p
!mv sift/* data/sift1M

In [ ]:
!pip install faiss-cpu -qqq

In [ ]:
import psutil

# 현재 프로세스가 사용하는 메모리 사용량을 메가바이트 단위로 반환.
def get_memory_usage_mb():
    process = psutil.Process()
    memory_info = process.memory_info()
    return memory_info.rss / (1024 * 1024)

import time
import faiss
from faiss.contrib.datasets import DatasetSIFT1M

ds = DatasetSIFT1M()

xq = ds.get_queries() # 검색 데이터
xb = ds.get_database() # 벡터 데이터
gt = ds.get_groundtruth() # 정답 데이터

In [ ]:
# 데이터가 늘어날 때 색인, 검색 시간, 메모리 사용량 변화.
k = 1
d = xq.shape[1]
nq = 1000 # 검색할 데이터 수
xq = xq[:nq]

for i in range(1, 10, 2):
    start_memory = get_memory_usage_mb()
    start_indexing = time.time()
    index = faiss.IndexFlatL2(d)
    index.add(xb[:(i+1)*100000])
    end_indexing = time.time()
    end_memory = get_memory_usage_mb()

    t0 = time.time()
    D, I = index.search(xq, k)
    t1 = time.time()
    print(f"데이터 {(i+1)*100000}개:")
    print(f"색인: {(end_indexing - start_indexing) * 1000 :.3f} ms ({end_memory - start_memory:.3f} MB) 검색: {(t1 - t0) * 1000 / nq :.3f} ms")

## ANN(Approximate Nearest Neighbor, 근사 최근접 이웃) 검색


*   (-) 정확도 일정 부분 희생.
*   (+) 속도 빠름.

In [ ]:
# HNSW(Hierarchical Navigable Small World)
import numpy as np

k = 1
d = xq.shape[1]
nq = 1000
xq = xq[:nq]

# m: 임베딩 벡터에 연결하는 간선 수.
# ㄴ 클수록
# ㄴㄴ (+) 재현율 증가
# ㄴㄴ (-) 메모리 사용량, 색인/검색 시간 증가.
for m in[8, 16, 32, 64]:
    index = faiss.IndexHNSWFlat(d, m)
    time.sleep(3)
    start_memory = get_memory_usage_mb()
    start_index = time.time()
    index.add(xb)
    end_memory = get_memory_usage_mb()
    end_index = time.time()
    print(f'M: {m} - 색인 시간: {end_index - start_index} s, 메모리 사용량: {end_memory - start_memory} MB')

    t0 = time.time()
    D, I = index.search(xq, k)
    t1 = time.time()

    recall_at_1 = np.equal(I, gt[:nq, :1]).sum() / float(nq)
    print(f'{(t1 - t0) * 1000.0 / nq:.3f} ms per query, R@1 {recall_at_1:.3f}')

In [ ]:
k = 1
d = xq.shape[1]
nq = 1000
xq = xq[:nq]

# ef_constrction: 색인 단계에서 후보군 크기 결정.
# ㄴ 클수록
# ㄴㄴ (+) 재현율 증가
# ㄴㄴ (-) 색인 시간 증가.
# ㄴㄴ 메모리 사용량, 검색 시간 영향 X
for ef_constrction in [40, 80, 160, 320]:
    index = faiss.IndexHNSWFlat(d, 32)
    index.hnsw.efConstruction = ef_constrction
    time.sleep(3)
    start_memory = get_memory_usage_mb()
    start_index = time.time()
    index.add(xb)
    end_memory = get_memory_usage_mb()
    end_index = time.time()
    print(f'efConstrction: {ef_constrction} - 색인 시간: {end_index - start_index} s, 메모리 사용량: {end_memory - start_memory} MB')

    t0 = time.time()
    D, I = index.search(xq, k)
    t1 = time.time()

    recall_at_1 = np.equal(I, gt[:nq, :1]).sum() / float(nq)
    print(f'{(t1 - t0) * 1000.0 / nq:.3f} ms per query, R@1 {recall_at_1:.3f}')

In [ ]:
# ef_search: 검색 단계에서 후보군 크기 결정.
# ㄴ 클수록
# ㄴㄴ (+) 재현율 증가
# ㄴㄴ (-) 검색 시간 증가.
# ㄴㄴ 메모리 사용량, 색인 시간 영향 X
for ef_search in [16, 32, 64, 128]:
    index.hnsw.efSearch = ef_search
    t0 = time.time()
    D, I = index.search(xq, k)
    t1 = time.time()

    recall_at_1 = np.equal(I, gt[:nq, :1]).sum() / float(nq)
    print(f'{(t1 - t0) * 1000.0 / nq:.3f} ms per query, R@1 {recall_at_1:.3f}')

In [ ]:
!pip install pinecone -qqq

In [ ]:
from pinecone import Pinecone, ServerlessSpec

pinecone_api_key = 'pinecone-api-key'
pc = Pinecone(api_key=pinecone_api_key)
# pc.create_index('llm-book', spec=ServerlessSpec('aws', 'us-east-1'), dimension=768)
index = pc.Index('llm-book') # 사용할 인덱스 불러옴.

In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

sentence_model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS') # 한국어 임베딩 모델 불러옴.
klue_dp_train = load_dataset('klue', 'dp', split='train[:100]')
embeddings = sentence_model.encode(klue_dp_train['sentence']) # 임베딩으로 변환.

In [ ]:
embeddings = embeddings.tolist()
insert_data = []
for idx, (embedding, text) in enumerate(zip(embeddings, klue_dp_train['sentence'])):
    insert_data.append({
        'id': str(idx),
        'values': embedding,
        'metadata': {'text': text}})

In [ ]:
upsert_response = index.upsert(vectors=insert_data, namespace='llm-book-sub') # 파인콘은 1개의 인덱스 안에 N개의 네임스페이스 구분 가능.

In [ ]:
# 인덱스 검색
query_response = index.query(
    namespace='llm-book-sub',
    top_k=10, # 결과 반환 개수

    # 검색 결과에 포함시킬 데이터 종류.
    include_values=True,
    include_metadata=True,

    vector=embeddings[0]
)
query_response

In [ ]:
# 문서 ID를 기반으로 데이터 수정 및 삭제
new_text = '변경할 새로운 텍스트'
new_embedding = sentence_model.encode(new_text).tolist()
update_response = index.update(
    id='doc_id',
    values=new_embedding,
    set_metadata={'text': new_text},
    namespace='llm-book-sub'
)

delete_response = index.delete(ids=['doc_id'], namespace='llm-book-sub')